In [1]:

import os
# Make sure you are in scMEDAL_for_scRNAseq dir
os.chdir("/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq")

from models.models import train_model_on_named_experiment

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [ ]:

os.getcwd()

## Training Models:

#### Implemented Models:
| Model Class | Model Description | 
|------------|--------------------|  
| `AE` | Autoencoder | 
| `AEC`| Autoencoder Classifier |
| `scMEDAL-FE` | Domain Adversarial Autoencoder |
| `scMEDAL-FEC`| Domain Adversarial Autoencoder Classifier |
| `scMEDAL-RE`| Domain Enhancing Autoencoder Classifier | 

#### Named Experiments:
| Valid Named Experiment | Dataset |  n_clusters | n_pred |
|------------------------|---------|-------------|--------|
| `AML`| Acute Myeloid Leukemia | 19 | 21 |
| `ASD`| Autism Spectrum Disorder | 31 | 17 | 
| `HH` | Healthy Heart | 147 | 13 | 

**Note:** If training on other datasets, the configs will need to be passed in as dictionaries to `model_kwargs` and `train_kwargs`.

`quick` is a boolean flag that can be passed to `train_kwargs` which shortens training to only 1 fold for 3 epochs.

In [ ]:
# Run "scmedalfe" and "scmedalre" models
scmedalfe_aml= train_model_on_named_experiment("scMEDAL-FE", "AML", model_kwargs={"n_latent_dims":50},train_kwargs={"quick":True}) #train_kwargs={"epochs": 2,"fold_list":[1,2]})
scmedalre_aml = train_model_on_named_experiment("scMEDAL-RE", "AML", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True})

## Analyzing Models:

In [2]:

# Update output paths of the models that you just run
from utils.defaults import AML_OUTPUTS_DIR
import os
model_folder_dict = {
    #"ae":"",
    #"ae":"",
    #"aec":"",
    "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_11-59",
    #"scmedalfec":"",
    "scmedalre":"run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-50_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_11-59",
}
model_paths = {k:os.path.join(AML_OUTPUTS_DIR, "latent_space", "log_transformed_2916hvggenes",k, v) for k, v in model_folder_dict.items()}


In [ ]:
# Run MEC on scmedalfe + scmedalre, target: 21 celltypes 
mec_aml = train_model_on_named_experiment("MEC", "AML", 
                                    train_kwargs={"quick":True, "results_path_dict":model_paths, },
                                    model_kwargs={"n_pred":21,"models_list":list(model_paths.keys()),
                                                   "latent_keys_config":{"fe_latent":"scmedalfe_latent", "re_latent":"scmedalre_latent"}})
                                                # "get_pca":True,
                                                #"latent_keys_config":{"fe_latent":"X_pca", "re_latent":"re_latent"}}

In [3]:
# Build the analysis object once
import analysis.analysis as aa

analysis_name = "AML_default"

# Compile clustering scores
# Note: if you only run 1 fold, you will see NAN in CI entries
aml = aa.AMLAnalysis(model_folder_dict, analysis_name)
res= aml.clustering_scores(model_folder_dict)
res

Directory created or already exists: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/AML/compare_models/log_transformed_2916hvggenes/AML_default

Computed aggregated scores DataFrame
Computed Confidence interval results for sample size: 10000, saved to: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/AML/compare_models/log_transformed_2916hvggenes/AML_default
 


batch                                                          \
              1/db                         ch                   silhouette   
              mean CI_lower CI_upper     mean CI_lower CI_upper       mean   
dataset_type                                                                 
scmedalfe      0.1      NaN      NaN   126.04      NaN      NaN      -0.19   
scmedalre      0.6      NaN      NaN  2026.11      NaN      NaN       0.44   

                               celltype                                      \
                                   1/db                         ch            
             CI_lower CI_upper     mean CI_lower CI_upper     mean CI_lower   
dataset_type                                                                  
scmedalfe         NaN      NaN     0.17      NaN      NaN  1128.00      NaN   
scmedalre         NaN      NaN     0.19      NaN      NaN   192.42      NaN   

                                                    
                      silhouette                    
             CI_upper       mean CI_lower CI_upper  
dataset_type                                        
scmedalfe         NaN      -0.02      NaN      NaN  
scmedalre         NaN      -0.05      NaN      NaN

In [ ]:
# Run genomaps on a subsample input+scmedalfe recon +scmedal re recon for different batches
batches = ["AML420B", "BM5", "MUTZ3"]
celltype = ["Mono", "Mono-like"]
# for this genomap config, you need to compute first the reconstructions from scMEDAL-FE and scMEDAL-RE
selected_models = {k: model_folder_dict[k] for k in ("scmedalfe", "scmedalre")}
gfd = aml.genomap(selected_models,
                            n_batches = 19,
                            num_iter = 2, # for quick test, otherwise 100
                            cell_id_col = "Cell",
                            gene_index_col = "Gene",
                            celltype = ["Mono", "Mono-like"],
                            batches=batches,
                            models=['scmedalre'],#if add_inputs_fe=True-> scmedalfe+ inputs are used for genomap creation by default, no need to add the to the list, 
                            types=["train"], 
                            splits=[1],
                            add_inputs_fe= True,
                            extra_recon = "fe",
                            min_val = -1,
                            max_val =2)

In [ ]:
# Plot umaps
processors = aml.umap(model_folder_dict, types=["train"], splits=[1])
processors